In [ ]:
import cv2
import numpy as np
import hashlib
import matplotlib.pyplot as plt

# -------------------------------
# Master Key (ONLY secret input)
# -------------------------------
MASTER_KEY = "PatientID_StudyID_PrivateSecret"

# -------------------------------
# Load and resize image
# -------------------------------

image_path = r"C:\Users\rajar\OneDrive\Documents\win sem 25-26\ecg.jpg"
img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
img = cv2.resize(img, (1024,1024), interpolation=cv2.INTER_CUBIC)
img = img.astype(np.uint8)

# -------------------------------
# Display Original Image
# -------------------------------
plt.figure(figsize=(4,4))
plt.imshow(img, cmap='gray')
plt.title("Original Grayscale Image")
plt.axis('off')
plt.show()

# -------------------------------
# Basic Info
# -------------------------------
print("Image shape:", img.shape)
print("Data type:", img.dtype)
print("Pixel range:", img.min(), "-", img.max())


In [ ]:
# -------------------------------
# Key Derivation Function (KDF)
# -------------------------------
def kdf(master_key, context, out_len=32):
    """
    Deterministic key derivation using SHA-256.
    Same (master_key + context) always gives the same output.
    """
    data = (master_key + context).encode()
    output = bytearray()
    current = hashlib.sha256(data).digest()

    while len(output) < out_len:
        current = hashlib.sha256(current + data).digest()
        output.extend(current)

    return bytes(output[:out_len])


# -------------------------------
# SHA-256 based keystream generator
# -------------------------------
def sha256_stream(key_bytes, length):
    """
    Generates a deterministic pseudo-random byte stream
    using repeated SHA-256 hashing.
    """
    out = bytearray()
    counter = 0

    while len(out) < length:
        data = key_bytes + counter.to_bytes(4, 'big')
        out.extend(hashlib.sha256(data).digest())
        counter += 1

    return np.frombuffer(out[:length], dtype=np.uint8)


# -------------------------------
# Layer-level keys
# -------------------------------
K_L1 = kdf(MASTER_KEY, "LAYER1_KEY", 32)   # Coarse layer
K_L2 = kdf(MASTER_KEY, "LAYER2_KEY", 32)   # Detail layer
K_L3 = kdf(MASTER_KEY, "LAYER3_KEY", 32)   # Fine layer

# -------------------------------
# Global diffusion key
# -------------------------------
K_GLOBAL = kdf(MASTER_KEY, "GLOBAL_DIFFUSION", 32)


print("Layer keys:")
print("  Layer-1 key hash :", hashlib.sha256(K_L1).hexdigest()[:12])
print("  Layer-2 key hash :", hashlib.sha256(K_L2).hexdigest()[:12])
print("  Layer-3 key hash :", hashlib.sha256(K_L3).hexdigest()[:12])
print("  Global key hash  :", hashlib.sha256(K_GLOBAL).hexdigest()[:12])


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# -------------------------------
# Load grayscale medical image
# -------------------------------
image_path = r"C:\Users\rajar\OneDrive\Documents\win sem 25-26\ecg.jpg"
img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

# Resize for uniform processing
img = cv2.resize(img, (1024,1024), interpolation=cv2.INTER_CUBIC)
img = img.astype(np.uint8)

# -------------------------------
# Bit-plane extraction
# -------------------------------
def extract_bit_plane(image, bit):
    return ((image >> bit) & 1).astype(np.uint8) << bit

# -------------------------------
# Extract all 8 bit-planes
# -------------------------------
bit_planes = {}
for b in range(8):
    bit_planes[b] = extract_bit_plane(img, b)

# -------------------------------
# Display bit-planes in one row (BP7 to BP0)
# -------------------------------
plt.figure(figsize=(16, 3))

for i, b in enumerate(range(7, -1, -1)):
    plt.subplot(1, 8, i + 1)
    plt.imshow(bit_planes[b], cmap='gray')
    plt.title(f"BP{b}", fontsize=18)
    plt.axis('off')

plt.tight_layout()
plt.show()


for b in range(7, -1, -1):
    print(f"BP{b} pixel range:", bit_planes[b].min(), "-", bit_planes[b].max())

In [ ]:
# -------------------------------
# Semantic layer grouping (TRUE content)
# -------------------------------
# Layer-1: Coarse / MSB (structural information)
L1_true = bit_planes[7] | bit_planes[6]

# Layer-2: Detail / Middle bits (edges, texture)
L2_true = bit_planes[5] | bit_planes[4] | bit_planes[3]

# Layer-3: Fine / LSB (very fine variations)
L3_true = bit_planes[2] | bit_planes[1] | bit_planes[0]

plt.figure(figsize=(16,5))  

plt.subplot(1,3,1)
plt.imshow(L1_true, cmap='gray')
plt.title("Layer-1 (BP7 + BP6)", fontsize=15, pad=10)
plt.axis('off')

plt.subplot(1,3,2)
plt.imshow(L2_true, cmap='gray')
plt.title("Layer-2 (BP5 + BP4 + BP3)", fontsize=15, pad=10)
plt.axis('off')

plt.subplot(1,3,3)
plt.imshow(L3_true, cmap='gray')
plt.title("Layer-3 (BP2 + BP1 + BP0)", fontsize=15, pad=10)
plt.axis('off')

plt.subplots_adjust(wspace=0.2, top=0.85)
plt.show()




In [ ]:
# -------------------------------
# AES S-box (fixed base)
# -------------------------------
AES_SBOX = np.array([
    99,124,119,123,242,107,111,197,48,1,103,43,254,215,171,118,
    202,130,201,125,250,89,71,240,173,212,162,175,156,164,114,192,
    183,253,147,38,54,63,247,204,52,165,229,241,113,216,49,21,
    4,199,35,195,24,150,5,154,7,18,128,226,235,39,178,117,
    9,131,44,26,27,110,90,160,82,59,214,179,41,227,47,132,
    83,209,0,237,32,252,177,91,106,203,190,57,74,76,88,207,
    208,239,170,251,67,77,51,133,69,249,2,127,80,60,159,168,
    81,163,64,143,146,157,56,245,188,182,218,33,16,255,243,210,
    205,12,19,236,95,151,68,23,196,167,126,61,100,93,25,115,
    96,129,79,220,34,42,144,136,70,238,184,20,222,94,11,219,
    224,50,58,10,73,6,36,92,194,211,172,98,145,149,228,121,
    231,200,55,109,141,213,78,169,108,86,244,234,101,122,174,8,
    186,120,37,46,28,166,180,198,232,221,116,31,75,189,139,138,
    112,62,181,102,72,3,246,14,97,53,87,185,134,193,29,158,
    225,248,152,17,105,217,142,148,155,30,135,233,206,85,40,223,
    140,161,137,13,191,230,66,104,65,153,45,15,176,84,187,22
], dtype=np.uint8)


# -------------------------------
# Dynamic S-box generation
# -------------------------------
def generate_dynamic_sbox(key_bytes):
    seed = int.from_bytes(hashlib.sha256(key_bytes).digest()[:4], "big")
    rng = np.random.default_rng(seed)
    sbox = AES_SBOX.copy()
    rng.shuffle(sbox)
    return sbox


SBOX_L1 = generate_dynamic_sbox(K_L1)

INV_SBOX_L1 = np.zeros(256, dtype=np.uint8)
for i in range(256):
    INV_SBOX_L1[SBOX_L1[i]] = i


# -------------------------------
# SubBytes
# -------------------------------
L1_sub = SBOX_L1[L1_true]


# -------------------------------
# Round keys
# -------------------------------
h, w = L1_sub.shape
rng = np.random.default_rng(int.from_bytes(K_L1[:4], "big"))
round_keys = rng.integers(0, 256, size=h * w, dtype=np.uint8)


# -------------------------------
# AES-like block encryption
# -------------------------------
L1_blk = np.zeros_like(L1_sub, dtype=np.uint8)
idx = 0

for i in range(0, h, 4):
    for j in range(0, w, 4):
        block = L1_sub[i:i+4, j:j+4].copy()
        if block.shape != (4, 4):
            continue

        rk = round_keys[idx:idx+16].reshape(4, 4)
        idx += 16
        block ^= rk

        for r in range(4):
            block[r,1] ^= block[r,0]
            block[r,2] ^= block[r,1]
            block[r,3] ^= block[r,2]

        for c in range(4):
            block[1,c] ^= block[0,c]
            block[2,c] ^= block[1,c]
            block[3,c] ^= block[2,c]

        L1_blk[i:i+4, j:j+4] = block


# -------------------------------
# Pixel-chained diffusion
# -------------------------------
rng = np.random.default_rng(int.from_bytes(K_L1[4:8], "big"))
diff_seq = rng.integers(0, 256, size=h * w, dtype=np.uint8).reshape(h, w)

L1_diff = np.zeros_like(L1_blk, dtype=np.uint8)
prev = 0
for i in range(h):
    for j in range(w):
        L1_diff[i, j] = L1_blk[i, j] ^ prev ^ diff_seq[i, j]
        prev = L1_diff[i, j]


# -------------------------------
# Final XOR stream
# -------------------------------
stream = sha256_stream(K_L1, h * w).reshape(h, w)
L1_enc = L1_diff ^ stream


# ===============================
# Layer-1 Decryption
# ===============================

L1_diff_dec = L1_enc ^ stream

L1_blk_dec = np.zeros_like(L1_diff_dec, dtype=np.uint8)
prev = 0
for i in range(h):
    for j in range(w):
        L1_blk_dec[i, j] = L1_diff_dec[i, j] ^ prev ^ diff_seq[i, j]
        prev = L1_diff_dec[i, j]

L1_sub_dec = np.zeros_like(L1_blk_dec, dtype=np.uint8)
idx = 0

for i in range(0, h, 4):
    for j in range(0, w, 4):
        block = L1_blk_dec[i:i+4, j:j+4].copy()
        if block.shape != (4, 4):
            continue

        for c in range(4):
            block[3,c] ^= block[2,c]
            block[2,c] ^= block[1,c]
            block[1,c] ^= block[0,c]

        for r in range(4):
            block[r,3] ^= block[r,2]
            block[r,2] ^= block[r,1]
            block[r,1] ^= block[r,0]

        rk = round_keys[idx:idx+16].reshape(4, 4)
        idx += 16
        block ^= rk

        L1_sub_dec[i:i+4, j:j+4] = block

L1_dec = INV_SBOX_L1[L1_sub_dec]


# -------------------------------
# Visualization
# -------------------------------
plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.imshow(L1_true, cmap='gray')
plt.title("Layer-1 TRUE (Original)")
plt.axis('off')

plt.subplot(1,3,2)
plt.imshow(L1_enc, cmap='gray')
plt.title("Layer-1 Encrypted")
plt.axis('off')

plt.subplot(1,3,3)
plt.imshow(L1_dec, cmap='gray')
plt.title("Layer-1 Decrypted")
plt.axis('off')

plt.tight_layout()
plt.show()


# -------------------------------
# Verification
# -------------------------------
print("Layer-1 TRUE perfect reconstruction:",
      np.array_equal(L1_true, L1_dec))


In [ ]:
# -------------------------------
# Generate key-dependent S-box (Layer-2)
# -------------------------------
def generate_sbox_L2(key_bytes):
    seed = int.from_bytes(hashlib.sha256(key_bytes).digest()[:4], "big")
    rng = np.random.default_rng(seed)

    sbox = np.arange(256, dtype=np.uint8)
    rng.shuffle(sbox)

    inv_sbox = np.zeros(256, dtype=np.uint8)
    for i in range(256):
        inv_sbox[sbox[i]] = i

    return sbox, inv_sbox


SBOX_L2, INV_SBOX_L2 = generate_sbox_L2(K_L2)


# -------------------------------
# Round keys (Layer-2)
# -------------------------------
h2, w2 = L2_true.shape
rng = np.random.default_rng(int.from_bytes(K_L2[:4], "big"))
rk_L2 = rng.integers(0, 256, size=h2 * w2, dtype=np.uint8)


# ===============================
# Layer-2 Encryption
# ===============================
L2_enc = np.zeros_like(L2_true, dtype=np.uint8)
idx = 0

for i in range(0, h2, 4):
    for j in range(0, w2, 4):
        block = L2_true[i:i+4, j:j+4].copy()
        if block.shape != (4, 4):
            continue

        # SubBytes
        block = SBOX_L2[block]

        # AddRoundKey
        block ^= rk_L2[idx:idx+16].reshape(4, 4)
        idx += 16

        # Row diffusion
        for r in range(4):
            block[r,1] ^= block[r,0]
            block[r,2] ^= block[r,1]
            block[r,3] ^= block[r,2]

        # ShiftColumns (invertible)
        block = np.roll(block, 1, axis=1)

        L2_enc[i:i+4, j:j+4] = block


# ===============================
# Layer-2 Decryption
# ===============================
L2_dec = np.zeros_like(L2_enc, dtype=np.uint8)
idx = 0

for i in range(0, h2, 4):
    for j in range(0, w2, 4):
        block = L2_enc[i:i+4, j:j+4].copy()
        if block.shape != (4, 4):
            continue

        # Inverse ShiftColumns
        block = np.roll(block, -1, axis=1)

        # Inverse row diffusion
        for r in range(4):
            block[r,3] ^= block[r,2]
            block[r,2] ^= block[r,1]
            block[r,1] ^= block[r,0]

        # Inverse AddRoundKey
        block ^= rk_L2[idx:idx+16].reshape(4, 4)
        idx += 16

        # Inverse SubBytes
        block = INV_SBOX_L2[block]

        L2_dec[i:i+4, j:j+4] = block


# -------------------------------
# Visualization
# -------------------------------
plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.imshow(L2_true, cmap='gray')
plt.title("Layer-2 TRUE (Original)")
plt.axis('off')

plt.subplot(1,3,2)
plt.imshow(L2_enc, cmap='gray')
plt.title("Layer-2 Encrypted")
plt.axis('off')

plt.subplot(1,3,3)
plt.imshow(L2_dec, cmap='gray')
plt.title("Layer-2 Decrypted")
plt.axis('off')

plt.tight_layout()
plt.show()


# -------------------------------
# Verification
# -------------------------------
print("Layer-2 TRUE perfect reconstruction:",
      np.array_equal(L2_true, L2_dec))


In [ ]:
# -------------------------------
# Keystream for Layer-3
# -------------------------------
h3, w3 = L3_true.shape
stream_L3 = sha256_stream(K_L3, h3 * w3).reshape(h3, w3)


# ===============================
# Layer-3 Encryption
# ===============================
L3_enc = L3_true ^ stream_L3


# ===============================
# Layer-3 Decryption
# ===============================
L3_dec = L3_enc ^ stream_L3


# -------------------------------
# Visualization
# -------------------------------
plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.imshow(L3_true, cmap='gray')
plt.title("Layer-3 TRUE (Original)")
plt.axis('off')

plt.subplot(1,3,2)
plt.imshow(L3_enc, cmap='gray')
plt.title("Layer-3 Encrypted")
plt.axis('off')

plt.subplot(1,3,3)
plt.imshow(L3_dec, cmap='gray')
plt.title("Layer-3 Decrypted")
plt.axis('off')

plt.tight_layout()
plt.show()


# -------------------------------
# Verification
# -------------------------------
print("Layer-3 TRUE perfect reconstruction:",
      np.array_equal(L3_true, L3_dec))


In [ ]:
# -------------------------------
# Recombine encrypted semantic layers
# -------------------------------
img_enc_layers = L1_enc ^ L2_enc ^ L3_enc


# -------------------------------
# Global diffusion (final encryption stage)
# -------------------------------
def global_diffusion_encrypt(img, key_bytes):
    h, w = img.shape
    out = np.zeros_like(img, dtype=np.uint8)

    # ---- KEYED + PLAINTEXT-DEPENDENT SEED ----
    img_hash = hashlib.sha256(img.tobytes()).digest()
    seed_material = key_bytes + img_hash
    seed = int.from_bytes(hashlib.sha256(seed_material).digest()[:4], "big")

    rng = np.random.default_rng(seed)
    seq = rng.integers(0, 256, size=h * w, dtype=np.uint8).reshape(h, w)

    for i in range(h):
        for j in range(w):
            if i == 0 and j == 0:
                out[i, j] = img[i, j] ^ seq[i, j]
            elif j == 0:
                out[i, j] = img[i, j] ^ out[i-1, w-1] ^ seq[i, j]
            else:
                out[i, j] = img[i, j] ^ out[i, j-1] ^ seq[i, j]

    return out



# -------------------------------
# Final encrypted image
# -------------------------------
img_cipher = global_diffusion_encrypt(img_enc_layers, K_GLOBAL)


# -------------------------------
# Visualization
# -------------------------------
plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.imshow(img, cmap='gray')
plt.title("Original Image")
plt.axis('off')

plt.subplot(1,3,2)
plt.imshow(img_enc_layers, cmap='gray')
plt.title("Recombined Encrypted Layers")
plt.axis('off')

plt.subplot(1,3,3)
plt.imshow(img_cipher, cmap='gray')
plt.title("Final Encrypted Image")
plt.axis('off')

plt.tight_layout()
plt.show()


print("Encrypted image range:",
      img_cipher.min(), "-", img_cipher.max())


In [ ]:
# ===============================
# Global diffusion decryption
# ===============================
def global_diffusion_decrypt(cipher, key_bytes):
    h, w = cipher.shape
    out = np.zeros_like(cipher, dtype=np.uint8)

    seed = int.from_bytes(hashlib.sha256(key_bytes).digest()[:4], "big")
    rng = np.random.default_rng(seed)
    seq = rng.integers(0, 256, size=h * w, dtype=np.uint8).reshape(h, w)

    for i in range(h):
        for j in range(w):
            if i == 0 and j == 0:
                out[i, j] = cipher[i, j] ^ seq[i, j]
            elif j == 0:
                out[i, j] = cipher[i, j] ^ cipher[i-1, w-1] ^ seq[i, j]
            else:
                out[i, j] = cipher[i, j] ^ cipher[i, j-1] ^ seq[i, j]

    return out


# -------------------------------
# Undo global diffusion
# -------------------------------
img_enc_layers_dec = global_diffusion_decrypt(img_cipher, K_GLOBAL)


# -------------------------------
# Recover encrypted semantic layers
# -------------------------------
# Because recombination was XOR-based:
# img_enc_layers = L1_enc ⊕ L2_enc ⊕ L3_enc
L1_enc_rec = img_enc_layers_dec ^ L2_enc ^ L3_enc
L2_enc_rec = img_enc_layers_dec ^ L1_enc ^ L3_enc
L3_enc_rec = img_enc_layers_dec ^ L1_enc ^ L2_enc


# -------------------------------
# Reconstruct original image
# -------------------------------
img_reconstructed = (L1_dec | L2_dec | L3_dec).astype(np.uint8)


# -------------------------------
# Visualization
# -------------------------------
plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.imshow(img, cmap='gray')
plt.title("Original Image")
plt.axis('off')

plt.subplot(1,3,2)
plt.imshow(img_cipher, cmap='gray')
plt.title("Encrypted Image")
plt.axis('off')

plt.subplot(1,3,3)
plt.imshow(img_reconstructed, cmap='gray')
plt.title("Fully Decrypted Image")
plt.axis('off')

plt.tight_layout()
plt.show()


# -------------------------------
# Final verification
# -------------------------------
diff = np.abs(img.astype(int) - img_reconstructed.astype(int))
print("Perfect reconstruction of full image:",
      np.array_equal(img, img_reconstructed))
print("Max pixel difference:", diff.max())
print("Number of differing pixels:", np.count_nonzero(diff))


In [ ]:
# -------------------------------
# Entropy
# -------------------------------
def image_entropy(img):
    hist = np.bincount(img.flatten(), minlength=256)
    prob = hist / np.sum(hist)
    prob = prob[prob > 0]
    return -np.sum(prob * np.log2(prob))


# -------------------------------
# Correlation
# -------------------------------
def correlation(img, mode='horizontal'):
    h, w = img.shape
    if mode == 'horizontal':
        x = img[:, :-1].flatten()
        y = img[:, 1:].flatten()
    elif mode == 'vertical':
        x = img[:-1, :].flatten()
        y = img[1:, :].flatten()
    elif mode == 'diagonal':
        x = img[:-1, :-1].flatten()
        y = img[1:, 1:].flatten()
    return np.corrcoef(x, y)[0, 1]


# -------------------------------
# NPCR
# -------------------------------
def npcr(img1, img2):
    return np.sum(img1 != img2) / img1.size * 100


# -------------------------------
# UACI
# -------------------------------
def uaci(img1, img2):
    return np.mean(np.abs(img1.astype(np.int16) - img2.astype(np.int16)) / 255) * 100


# ============================================================
# Create modified image (1-bit change)
# ============================================================
img_mod = img.copy()
img_mod[0, 0] ^= 1


# ============================================================
# Rebuild semantic layers (modified image)
# ============================================================
bp_mod = {}
for b in range(8):
    bp_mod[b] = ((img_mod >> b) & 1).astype(np.uint8) << b

L1m = bp_mod[7] | bp_mod[6]
L2m = bp_mod[5] | bp_mod[4] | bp_mod[3]
L3m = bp_mod[2] | bp_mod[1] | bp_mod[0]


# ============================================================
# FULL re-encryption of modified image
# ============================================================

# ---- Layer-1 encryption ----
L1m_sub = SBOX_L1[L1m]

L1m_blk = np.zeros_like(L1m_sub)
idx = 0
for i in range(0, h, 4):
    for j in range(0, w, 4):
        block = L1m_sub[i:i+4, j:j+4].copy()
        if block.shape != (4,4):
            continue

        rk = round_keys[idx:idx+16].reshape(4,4)
        idx += 16
        block ^= rk

        for r in range(4):
            block[r,1] ^= block[r,0]
            block[r,2] ^= block[r,1]
            block[r,3] ^= block[r,2]

        for c in range(4):
            block[1,c] ^= block[0,c]
            block[2,c] ^= block[1,c]
            block[3,c] ^= block[2,c]

        L1m_blk[i:i+4, j:j+4] = block

L1m_diff = np.zeros_like(L1m_blk)
prev = 0
for i in range(h):
    for j in range(w):
        L1m_diff[i,j] = L1m_blk[i,j] ^ prev ^ diff_seq[i,j]
        prev = L1m_diff[i,j]

L1m_enc = L1m_diff ^ stream


# ---- Layer-2 encryption ----
L2m_enc = np.zeros_like(L2m)
idx = 0
for i in range(0, h2, 4):
    for j in range(0, w2, 4):
        block = L2m[i:i+4, j:j+4].copy()
        if block.shape != (4,4):
            continue

        block = SBOX_L2[block]
        block ^= rk_L2[idx:idx+16].reshape(4,4)
        idx += 16

        for r in range(4):
            block[r,1] ^= block[r,0]
            block[r,2] ^= block[r,1]
            block[r,3] ^= block[r,2]

        block = np.roll(block, 1, axis=1)
        L2m_enc[i:i+4, j:j+4] = block


# ---- Layer-3 encryption ----
L3m_enc = L3m ^ stream_L3


# ---- Final encrypted modified image ----
img_enc_mod = global_diffusion_encrypt(
    L1m_enc ^ L2m_enc ^ L3m_enc,
    K_GLOBAL
)


# ============================================================
# Metrics table
# ============================================================
layers = {
    "Layer-1 (Coarse)": (L1_true, L1_enc),
    "Layer-2 (Detail)": (L2_true, L2_enc),
    "Layer-3 (Fine)":   (L3_true, L3_enc),
    "Final Image":      (img_cipher, img_enc_mod)
}

print("{:<20} {:<10} {:<10} {:<10} {:<10} {:<10} {:<10}".format(
    "Layer", "Entropy", "Corr_H", "Corr_V", "Corr_D", "NPCR", "UACI"
))

for name, (orig, enc) in layers.items():
    print("{:<20} {:<10.4f} {:<10.4f} {:<10.4f} {:<10.4f} {:<10.2f} {:<10.2f}".format(
        name,
        image_entropy(enc),
        correlation(enc, 'horizontal'),
        correlation(enc, 'vertical'),
        correlation(enc, 'diagonal'),
        npcr(orig, enc),
        uaci(orig, enc)
    ))

In [ ]:
import matplotlib.pyplot as plt

# -------------------------------------------------
# Layers dictionary (CORRECT variable names)
# -------------------------------------------------
layer_dict = {
    "Layer-1": (L1_true, L1_enc, L1_dec),
    "Layer-2": (L2_true, L2_enc, L2_dec),
    "Layer-3":   (L3_true, L3_enc, L3_dec)
}

# Add full image as the last row
all_images = list(layer_dict.items()) + [
    ("Full Image", (img, img_cipher, img_reconstructed))
]

# -------------------------------------------------
# Create figure
# -------------------------------------------------
fig, axes = plt.subplots(
    nrows=len(all_images),
    ncols=3,
    figsize=(12, 4 * len(all_images))
)

# -------------------------------------------------
# Column titles
# -------------------------------------------------
col_titles = ["Original", "Encrypted", "Decrypted"]
for ax, title in zip(axes[0], col_titles):
    ax.set_title(title, fontsize=20, fontweight='bold')

# -------------------------------------------------
# Fill in images
# -------------------------------------------------
for row_idx, (name, (orig, enc, dec)) in enumerate(all_images):
    for col_idx, image in enumerate([orig, enc, dec]):
        ax = axes[row_idx, col_idx]
        ax.imshow(image, cmap='gray')
        ax.set_xticks([])
        ax.set_yticks([])

        # Row label on first column
        if col_idx == 0:
            ax.set_ylabel(
                name,
                fontsize=20,
                fontweight='bold',
                rotation=0,
                labelpad=60,
                va='center'
            )

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# -------------------------------
# Images for histogram
# -------------------------------
images = {
    "Original Image": img,
    "Encrypted Image": img_cipher,
    "Decrypted Image": img_reconstructed
}

# -------------------------------
# Plot histograms
# -------------------------------
plt.figure(figsize=(12,4))

for i, (title, image) in enumerate(images.items(), 1):
    plt.subplot(1, 3, i)
    plt.hist(image.flatten(), bins=256, range=(0,255))
    plt.title(title)
    plt.xlabel("Pixel Intensity")
    plt.ylabel("Frequency")
    plt.xlim(0,255)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# -------------------------------------------------
# Create figure
# -------------------------------------------------
fig = plt.figure(figsize=(28, 6))

# Tight spacing, images closer than histograms
gs = GridSpec(
    1, 4,
    width_ratios=[1, 1, 1, 1],
    wspace=0.15   # <-- KEY FIX (controls horizontal gap)
)

# -------------------------------------------------
# Original Image
# -------------------------------------------------
ax1 = fig.add_subplot(gs[0, 0])
ax1.imshow(img, cmap='gray')
ax1.set_title("Original Image", fontsize=18, fontweight='bold')
ax1.axis('off')

# -------------------------------------------------
# Encrypted Image
# -------------------------------------------------
ax2 = fig.add_subplot(gs[0, 1])
ax2.imshow(img_cipher, cmap='gray')
ax2.set_title("Encrypted Image", fontsize=18, fontweight='bold')
ax2.axis('off')

# -------------------------------------------------
# Original Histogram
# -------------------------------------------------
ax3 = fig.add_subplot(gs[0, 2])
ax3.hist(img.flatten(), bins=256, range=(0, 255))
ax3.set_title("Original Histogram", fontsize=18, fontweight='bold')
ax3.set_xlabel("Pixel Intensity", fontsize=14)
ax3.set_ylabel("Frequency", fontsize=14)
ax3.set_xlim(0, 255)

# -------------------------------------------------
# Encrypted Histogram
# -------------------------------------------------
ax4 = fig.add_subplot(gs[0, 3])
ax4.hist(img_cipher.flatten(), bins=256, range=(0, 255))
ax4.set_title("Encrypted Histogram", fontsize=18, fontweight='bold')
ax4.set_xlabel("Pixel Intensity", fontsize=14)
ax4.set_ylabel("Frequency", fontsize=14)
ax4.set_xlim(0, 255)

# IMPORTANT: do NOT use tight_layout here
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2

# =========================================================
# Load original image (grayscale)
# =========================================================
img_path = r"C:\Users\rajar\OneDrive\Documents\win sem 25-26\xray.jpeg"
img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
img = cv2.resize(img, (1024,1024), interpolation=cv2.INTER_CUBIC)
img = img.astype(np.uint8)

# img_cipher should already exist in your code
# (your encrypted image)

# =========================================================
# Function to get overall adjacent pixels (combined H+V+D)
# =========================================================
def get_overall_adjacent(img, num_samples=8000):
    h, w = img.shape
    
    # Horizontal
    x1 = img[:, :-1].flatten()
    y1 = img[:, 1:].flatten()
    
    # Vertical
    x2 = img[:-1, :].flatten()
    y2 = img[1:, :].flatten()
    
    # Diagonal
    x3 = img[:-1, :-1].flatten()
    y3 = img[1:, 1:].flatten()
    
    # Combine all
    x = np.concatenate([x1, x2, x3])
    y = np.concatenate([y1, y2, y3])
    
    idx = np.random.choice(len(x), num_samples, replace=False)
    return x[idx], y[idx]

# =========================================================
# Get data
# =========================================================
x_orig, y_orig = get_overall_adjacent(img)
x_enc, y_enc = get_overall_adjacent(img_cipher)

# =========================================================
# Plot
# =========================================================
plt.figure(figsize=(15,4))

# --- Original Image ---
plt.subplot(1,3,1)
plt.imshow(img, cmap='gray')
plt.title("Original XRAY Image", fontsize=16, fontweight='bold')
plt.axis('off')

# --- Original correlation ---
plt.subplot(1,3,2)
plt.scatter(x_orig, y_orig, s=1)
plt.title("Original Correlation", fontsize=16, fontweight='bold')
plt.xlim(0,255)
plt.ylim(0,255)

# --- Encrypted correlation ---
plt.subplot(1,3,3)
plt.scatter(x_enc, y_enc, s=1)
plt.title("Encrypted Correlation", fontsize=16, fontweight='bold')
plt.xlim(0,255)
plt.ylim(0,255)


plt.tight_layout()

# Save high quality figure
plt.savefig("overall_correlation.png", dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
import numpy as np
import hashlib
import matplotlib.pyplot as plt
import time


# -------------------------------
# AES S-box (fixed base)
# -------------------------------
AES_SBOX = np.array([
99,124,119,123,242,107,111,197,48,1,103,43,254,215,171,118,
202,130,201,125,250,89,71,240,173,212,162,175,156,164,114,192,
183,253,147,38,54,63,247,204,52,165,229,241,113,216,49,21,
4,199,35,195,24,150,5,154,7,18,128,226,235,39,178,117,
9,131,44,26,27,110,90,160,82,59,214,179,41,227,47,132,
83,209,0,237,32,252,177,91,106,203,190,57,74,76,88,207,
208,239,170,251,67,77,51,133,69,249,2,127,80,60,159,168,
81,163,64,143,146,157,56,245,188,182,218,33,16,255,243,210,
205,12,19,236,95,151,68,23,196,167,126,61,100,93,25,115,
96,129,79,220,34,42,144,136,70,238,184,20,222,94,11,219,
224,50,58,10,73,6,36,92,194,211,172,98,145,149,228,121,
231,200,55,109,141,213,78,169,108,86,244,234,101,122,174,8,
186,120,37,46,28,166,180,198,232,221,116,31,75,189,139,138,
112,62,181,102,72,3,246,14,97,53,87,185,134,193,29,158,
225,248,152,17,105,217,142,148,155,30,135,233,206,85,40,223,
140,161,137,13,191,230,66,104,65,153,45,15,176,84,187,22
], dtype=np.uint8)


# -------------------------------
# Dynamic S-box generation
# -------------------------------
def generate_dynamic_sbox(key_bytes):

    digest = hashlib.sha256(key_bytes).digest()

    c = digest[0]

    dynamic_sbox = np.zeros(256, dtype=np.uint8)

    for x in range(256):

        y = AES_SBOX[x]

        y ^= c
        y = ((y << 1) | (y >> 7)) & 0xFF
        y ^= digest[1]

        dynamic_sbox[x] = y

    if len(np.unique(dynamic_sbox)) != 256:
        return AES_SBOX.copy()

    return dynamic_sbox


SBOX_L1 = generate_dynamic_sbox(K_L1)


# -------------------------------
# Inverse S-box
# -------------------------------
INV_SBOX_L1 = np.zeros(256, dtype=np.uint8)
for i in range(256):
    INV_SBOX_L1[SBOX_L1[i]] = i


# =========================================================
# S-BOX SECURITY ANALYSIS
# =========================================================

print("S-box bijective:", len(np.unique(SBOX_L1)) == 256)


# -------------------------------
# Walsh Transform
# -------------------------------
def walsh_transform(f):

    n = len(f)
    W = np.copy(f) * 2 - 1
    h = 1

    while h < n:

        for i in range(0, n, h * 2):

            for j in range(i, i + h):

                x = W[j]
                y = W[j + h]

                W[j] = x + y
                W[j + h] = x - y

        h *= 2

    return W


# -------------------------------
# Nonlinearity
# -------------------------------
def compute_nonlinearity(sbox):

    nonlinearities = []

    for bit in range(8):

        f = [(sbox[i] >> bit) & 1 for i in range(256)]
        f = np.array(f)

        W = walsh_transform(f)
        max_w = np.max(np.abs(W))

        nl = (2**7) - (max_w / 2)
        nonlinearities.append(nl)

    return nonlinearities


nl = compute_nonlinearity(SBOX_L1)

print("Nonlinearity per output bit:", nl)
print("Average Nonlinearity:", np.mean(nl))


# -------------------------------
# Strict Avalanche Criterion
# -------------------------------
def compute_sac(sbox):

    sac_matrix = np.zeros((8,8))

    for input_bit in range(8):

        for x in range(256):

            flipped = x ^ (1 << input_bit)

            y1 = sbox[x]
            y2 = sbox[flipped]

            diff = y1 ^ y2

            for output_bit in range(8):

                if (diff >> output_bit) & 1:
                    sac_matrix[input_bit][output_bit] += 1

    sac_matrix = sac_matrix / 256

    return sac_matrix


sac = compute_sac(SBOX_L1)

print("Average SAC:", np.mean(sac))

# -------------------------------
def nonlinearity(f):

    f = np.array(f)
    f = 1 - 2*f   # convert {0,1} → {1,-1}

    W = np.fft.fft(f)   # Walsh spectrum approximation
    max_w = np.max(np.abs(W))

    nl = 128 - max_w/2
    return nl

# -------------------------------
# BIC Nonlinearity
# -------------------------------
def bic_nonlinearity(sbox):

    bic_nl = []

    for i in range(8):
        for j in range(i+1, 8):

            f = []

            for x in range(256):

                y = sbox[x]

                bi = (y >> i) & 1
                bj = (y >> j) & 1

                f.append(bi ^ bj)

            nl = nonlinearity(f)   # Walsh transform
            bic_nl.append(nl)

    return np.mean(bic_nl)


# Compute BIC Nonlinearity
bic_nl_value = bic_nonlinearity(SBOX_L1)

# Print result
print("BIC Nonlinearity:", bic_nl_value)



# -------------------------------
# BIC-SAC
# -------------------------------
def compute_bic_sac(sac_matrix):

    bic_sac = np.zeros((8,8))

    for i in range(8):
        for j in range(8):

            if i == j:
                continue

            bic_sac[i,j] = abs(sac_matrix[i,j] - 0.5)

    return bic_sac


bic_sac = compute_bic_sac(sac)

print("BIC-SAC:", np.mean(bic_sac))


# -------------------------------
# Differential Uniformity
# -------------------------------
def differential_uniformity(sbox):

    max_count = 0

    for dx in range(1,256):

        counts = {}

        for x in range(256):

            x2 = x ^ dx
            dy = sbox[x] ^ sbox[x2]

            counts[dy] = counts.get(dy,0) + 1

        max_count = max(max_count, max(counts.values()))

    return max_count


du = differential_uniformity(SBOX_L1)

print("Differential Uniformity:", du)


# -------------------------------
# Linear Approximation Probability
# -------------------------------
def compute_lap(sbox):

    max_bias = 0

    for a in range(1,256):
        for b in range(1,256):

            count = 0

            for x in range(256):

                input_parity = bin(a & x).count("1") % 2
                output_parity = bin(b & sbox[x]).count("1") % 2

                if input_parity == output_parity:
                    count += 1

            bias = abs(count - 128)

            if bias > max_bias:
                max_bias = bias

    lap = max_bias / 256

    return lap


lap = compute_lap(SBOX_L1)

print("LAP:", lap)


# -------------------------------
# Differential Approximation Probability
# -------------------------------
def compute_dap(sbox):

    max_prob = 0

    for dx in range(1,256):

        counts = {}

        for x in range(256):

            x2 = x ^ dx
            dy = sbox[x] ^ sbox[x2]

            counts[dy] = counts.get(dy,0) + 1

        prob = max(counts.values()) / 256

        if prob > max_prob:
            max_prob = prob

    return max_prob


dap = compute_dap(SBOX_L1)

print("DAP:", dap)



L1_sub = SBOX_L1[L1_true]

h, w = L1_sub.shape
rng = np.random.default_rng(int.from_bytes(K_L1[:4], "big"))
round_keys = rng.integers(0, 256, size=h * w, dtype=np.uint8)

L1_blk = np.zeros_like(L1_sub, dtype=np.uint8)
idx = 0

for i in range(0, h, 4):
    for j in range(0, w, 4):

        block = L1_sub[i:i+4, j:j+4].copy()

        if block.shape != (4, 4):
            continue

        rk = round_keys[idx:idx+16].reshape(4, 4)
        idx += 16

        block ^= rk

        for r in range(4):
            block[r,1] ^= block[r,0]
            block[r,2] ^= block[r,1]
            block[r,3] ^= block[r,2]

        for c in range(4):
            block[1,c] ^= block[0,c]
            block[2,c] ^= block[1,c]
            block[3,c] ^= block[2,c]

        L1_blk[i:i+4, j:j+4] = block


rng = np.random.default_rng(int.from_bytes(K_L1[4:8], "big"))
diff_seq = rng.integers(0, 256, size=h*w, dtype=np.uint8).reshape(h,w)

L1_diff = np.zeros_like(L1_blk, dtype=np.uint8)

prev = 0
for i in range(h):
    for j in range(w):

        L1_diff[i,j] = L1_blk[i,j] ^ prev ^ diff_seq[i,j]
        prev = L1_diff[i,j]


stream = sha256_stream(K_L1, h*w).reshape(h,w)

L1_enc = L1_diff ^ stream



L1_diff_dec = L1_enc ^ stream

L1_blk_dec = np.zeros_like(L1_diff_dec, dtype=np.uint8)

prev = 0
for i in range(h):
    for j in range(w):

        L1_blk_dec[i,j] = L1_diff_dec[i,j] ^ prev ^ diff_seq[i,j]
        prev = L1_diff_dec[i,j]


L1_sub_dec = np.zeros_like(L1_blk_dec, dtype=np.uint8)

idx = 0

for i in range(0, h, 4):
    for j in range(0, w, 4):

        block = L1_blk_dec[i:i+4, j:j+4].copy()

        if block.shape != (4,4):
            continue

        for c in range(4):
            block[3,c] ^= block[2,c]
            block[2,c] ^= block[1,c]
            block[1,c] ^= block[0,c]

        for r in range(4):
            block[r,3] ^= block[r,2]
            block[r,2] ^= block[r,1]
            block[r,1] ^= block[r,0]

        rk = round_keys[idx:idx+16].reshape(4,4)
        idx += 16

        block ^= rk

        L1_sub_dec[i:i+4, j:j+4] = block


L1_dec = INV_SBOX_L1[L1_sub_dec]




plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.imshow(L1_true, cmap='gray')
plt.title("Layer-1 TRUE (Original)")
plt.axis('off')

plt.subplot(1,3,2)
plt.imshow(L1_enc, cmap='gray')
plt.title("Layer-1 Encrypted")
plt.axis('off')

plt.subplot(1,3,3)
plt.imshow(L1_dec, cmap='gray')
plt.title("Layer-1 Decrypted")
plt.axis('off')

plt.tight_layout()
plt.show()


print("Layer-1 TRUE perfect reconstruction:",
      np.array_equal(L1_true, L1_dec))


In [ ]:
import numpy as np

# -------------------------------
# Local Shannon Entropy
# -------------------------------
def local_entropy(img, block_size=64):

    h, w = img.shape
    entropies = []

    for i in range(0, h, block_size):
        for j in range(0, w, block_size):

            block = img[i:i+block_size, j:j+block_size]

            # skip incomplete blocks
            if block.shape != (block_size, block_size):
                continue

            hist = np.bincount(block.flatten(), minlength=256)
            prob = hist / np.sum(hist)

            prob = prob[prob > 0]

            ent = -np.sum(prob * np.log2(prob))

            entropies.append(ent)

    entropies = np.array(entropies)

    print("Local Shannon Entropy Results")
    print("------------------------------")
    print("Average Local Entropy:", np.mean(entropies))
    print("Minimum Block Entropy:", np.min(entropies))
    print("Maximum Block Entropy:", np.max(entropies))


# -------------------------------
# Run on encrypted image
# -------------------------------
local_entropy(img_cipher)

In [ ]:
USE_LAYER1 = True
USE_LAYER2 = True
USE_LAYER3 = True
USE_GLOBAL_DIFFUSION = True

In [ ]:
import time
import numpy as np
import hashlib
import matplotlib.pyplot as plt


# -------------------------------
# Recombine encrypted semantic layers (original image)
# -------------------------------
combined = np.zeros_like(L1_enc)

if USE_LAYER1:
    combined ^= L1_enc

if USE_LAYER2:
    combined ^= L2_enc

if USE_LAYER3:
    combined ^= L3_enc


# -------------------------------
# Global diffusion (final encryption stage)
# -------------------------------
def global_diffusion_encrypt(img, key_bytes):

    h, w = img.shape
    out = np.zeros_like(img, dtype=np.uint8)

    img_hash = hashlib.sha256(img.tobytes()).digest()
    seed_material = key_bytes + img_hash
    seed = int.from_bytes(hashlib.sha256(seed_material).digest()[:4], "big")

    rng = np.random.default_rng(seed)
    seq = rng.integers(0,256,size=h*w,dtype=np.uint8).reshape(h,w)

    for i in range(h):
        for j in range(w):

            if i == 0 and j == 0:
                out[i,j] = img[i,j] ^ seq[i,j]

            elif j == 0:
                out[i,j] = img[i,j] ^ out[i-1,w-1] ^ seq[i,j]

            else:
                out[i,j] = img[i,j] ^ out[i,j-1] ^ seq[i,j]

    return out


# -------------------------------
# Encrypt ORIGINAL recombined layers
# -------------------------------
start_time = time.perf_counter()

if USE_GLOBAL_DIFFUSION:
    img_cipher = global_diffusion_encrypt(combined, K_GLOBAL)
else:
    img_cipher = combined.copy()

end_time = time.perf_counter()

print("Global Diffusion Encryption Time:", end_time-start_time,"seconds")


# ============================================================
# Create modified image (1-bit change)
# ============================================================

img_mod = img.copy()
img_mod[0,0] ^= 1


# ============================================================
# Rebuild bitplanes of modified image
# ============================================================

bp_mod = {}

for b in range(8):
    bp_mod[b] = ((img_mod >> b) & 1).astype(np.uint8) << b

L1m = bp_mod[7] | bp_mod[6]
L2m = bp_mod[5] | bp_mod[4] | bp_mod[3]
L3m = bp_mod[2] | bp_mod[1] | bp_mod[0]


# ============================================================
# Recombine modified layers
# ============================================================

combined_mod = np.zeros_like(L1m)

if USE_LAYER1:
    combined_mod ^= L1m

if USE_LAYER2:
    combined_mod ^= L2m

if USE_LAYER3:
    combined_mod ^= L3m


# ============================================================
# Encrypt modified recombined layers
# ============================================================

if USE_GLOBAL_DIFFUSION:
    img_enc_mod = global_diffusion_encrypt(combined_mod, K_GLOBAL)
else:
    img_enc_mod = combined_mod.copy()


# -------------------------------
# Visualization
# -------------------------------
plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.imshow(img,cmap='gray')
plt.title("Original Image")
plt.axis('off')

plt.subplot(1,3,2)
plt.imshow(combined,cmap='gray')
plt.title("Recombined Layers")
plt.axis('off')

plt.subplot(1,3,3)
plt.imshow(img_cipher,cmap='gray')
plt.title("Final Cipher Image")
plt.axis('off')

plt.tight_layout()
plt.show()


# -------------------------------
# Debug
# -------------------------------
print("Encrypted image range:",img_cipher.min(),"-",img_cipher.max())

In [ ]:
import pandas as pd

# ------------------------------------------------
# Function to recombine layers with ablation flags
# ------------------------------------------------
def recombine_layers(L1, L2, L3, use_l1, use_l2, use_l3):

    out = np.zeros_like(L1)

    if use_l1:
        out ^= L1
    if use_l2:
        out ^= L2
    if use_l3:
        out ^= L3

    return out


# ------------------------------------------------
# Run ablation experiments
# ------------------------------------------------
configs = [
    ("Full Model", True, True, True, True),
    ("No Global Diffusion", True, True, True, False),
    ("No Layer-1", False, True, True, True),
    ("No Layer-2", True, False, True, True),
    ("No Layer-3", True, True, False, True),
]

results = []

for name, l1, l2, l3, gd in configs:

    # recombine layers
    comb = recombine_layers(L1_enc, L2_enc, L3_enc, l1, l2, l3)

    # encrypt
    if gd:
        cipher = global_diffusion_encrypt(comb, K_GLOBAL)
    else:
        cipher = comb.copy()

    # metrics
    entropy_val = image_entropy(cipher)
    corr_val = correlation(cipher, 'horizontal')

    # NPCR/UACI
    if gd:
        cipher_mod = global_diffusion_encrypt(combined_mod, K_GLOBAL)
    else:
        cipher_mod = combined_mod.copy()

    npcr_val = npcr(cipher, cipher_mod)
    uaci_val = uaci(cipher, cipher_mod)

    results.append([name, entropy_val, corr_val, npcr_val, uaci_val])


# ------------------------------------------------
# Display table
# ------------------------------------------------
df = pd.DataFrame(results, columns=[
    "Configuration",
    "Entropy",
    "Correlation",
    "NPCR (%)",
    "UACI (%)"
])

print("\nAblation Study Results\n")
print(df.to_string(index=False))

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

paths = [
    r"C:\Users\rajar\OneDrive\Documents\win sem 25-26\xray.jpeg",
    r"C:\Users\rajar\OneDrive\Documents\win sem 25-26\ultrasound.png",
    r"C:\Users\rajar\OneDrive\Documents\win sem 25-26\mri.jpg",
    r"C:\Users\rajar\OneDrive\Documents\win sem 25-26\ecg.jpg"
]

names = ["X-ray", "Ultrasound", "MRI", "ECG"]

results = []

plt.figure(figsize=(12,12))

for i, path in enumerate(paths):

    # -----------------------------
    # Load image
    # -----------------------------
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (256,256))

    # -----------------------------
    # Create modified image
    # -----------------------------
    img_mod = img.copy()
    img_mod[0,0] = (img_mod[0,0] + 1) % 256

    # -----------------------------
    # Encrypt both
    # -----------------------------
    cipher1 = global_diffusion_encrypt(img, K_GLOBAL)
    cipher2 = global_diffusion_encrypt(img_mod, K_GLOBAL)

    # -----------------------------
    # CPA condition
    # -----------------------------
    plain_diff = np.bitwise_xor(img, img_mod)
    cipher_diff = np.bitwise_xor(cipher1, cipher2)

    condition = plain_diff != cipher_diff

    changed_pixels = np.sum(condition)
    percentage = (changed_pixels / img.size) * 100

    results.append([names[i], changed_pixels, percentage])

    # -----------------------------
    # Plot images (4x4 layout)
    # -----------------------------
    plt.subplot(4,4,i*4+1)
    plt.imshow(img, cmap='gray')
    plt.axis('off')
    if i == 0:
        plt.title("Original (I1)")

    plt.subplot(4,4,i*4+2)
    plt.imshow(img_mod, cmap='gray')
    plt.axis('off')
    if i == 0:
        plt.title("Modified (I2)")

    plt.subplot(4,4,i*4+3)
    plt.imshow(cipher1, cmap='gray')
    plt.axis('off')
    if i == 0:
        plt.title("Cipher (C1)")

    plt.subplot(4,4,i*4+4)
    plt.imshow(cipher2, cmap='gray')
    plt.axis('off')
    if i == 0:
        plt.title("Modified Cipher (C2)")

plt.tight_layout()
plt.show()


# -----------------------------
# Table output
# -----------------------------
df = pd.DataFrame(results, columns=[
    "Image",
    "Pixels satisfying CPA condition",
    "Percentage (%)"
])

print(df)

In [ ]:
import numpy as np
import pandas as pd
import cv2
from skimage.metrics import structural_similarity as ssim
import hashlib

# =============================================================================
# NOISE ATTACK FUNCTIONS
# =============================================================================
def add_salt_pepper_noise(img, noise_ratio=0.05):
    noisy = img.copy()
    num_noise = int(noise_ratio * img.size)
    salt_coords = np.random.choice(img.size, num_noise//2, replace=False)
    noisy.flat[salt_coords] = 255
    pepper_coords = np.random.choice(img.size, num_noise//2, replace=False)
    noisy.flat[pepper_coords] = 0
    return noisy

def add_gaussian_noise(img, mean=0, sigma=20):
    noise = np.random.normal(mean, sigma, img.shape)
    noisy = np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    return noisy

def jpeg_compression_attack(img, quality=50):
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), quality]
    _, enc_jpg = cv2.imencode('.jpg', img, encode_param)
    img_jpg = cv2.imdecode(enc_jpg, cv2.IMREAD_GRAYSCALE)
    return img_jpg

def crop_attack(img, crop_ratio=0.1):
    h, w = img.shape
    crop_size = int(crop_ratio * min(h, w))
    start_y = np.random.randint(0, h - crop_size)
    start_x = np.random.randint(0, w - crop_size)
    cropped = img.copy()
    cropped[start_y:start_y+crop_size, start_x:start_x+crop_size] = 128
    return cropped

# =============================================================================
# DECRYPTION FUNCTIONS (UNCHANGED)
# =============================================================================
def decrypt_layer1(enc_img):
    h, w = enc_img.shape
    stream = sha256_stream(K_L1, h*w).reshape(h, w)
    L1_diff_dec = enc_img ^ stream

    L1_blk_dec = np.zeros_like(L1_diff_dec, dtype=np.uint8)
    prev = 0
    for i in range(h):
        for j in range(w):
            L1_blk_dec[i, j] = L1_diff_dec[i, j] ^ prev ^ diff_seq[i, j]
            prev = L1_diff_dec[i, j]

    L1_sub_dec = np.zeros_like(L1_blk_dec, dtype=np.uint8)
    idx = 0
    for i in range(0, h, 4):
        for j in range(0, w, 4):
            block = L1_blk_dec[i:i+4, j:j+4].copy()
            if block.shape != (4, 4):
                continue

            for c in range(4):
                block[3, c] ^= block[2, c]
                block[2, c] ^= block[1, c]
                block[1, c] ^= block[0, c]

            for r in range(4):
                block[r, 3] ^= block[r, 2]
                block[r, 2] ^= block[r, 1]
                block[r, 1] ^= block[r, 0]

            rk = round_keys[idx:idx+16].reshape(4, 4)
            idx += 16
            block ^= rk

            L1_sub_dec[i:i+4, j:j+4] = block

    return INV_SBOX_L1[L1_sub_dec]


def decrypt_layer2(enc_img):
    h2, w2 = enc_img.shape
    L2_dec = np.zeros_like(enc_img, dtype=np.uint8)
    idx = 0

    for i in range(0, h2, 4):
        for j in range(0, w2, 4):
            block = enc_img[i:i+4, j:j+4].copy()
            if block.shape != (4, 4):
                continue

            block = np.roll(block, -1, axis=1)

            for r in range(4):
                block[r, 3] ^= block[r, 2]
                block[r, 2] ^= block[r, 1]
                block[r, 1] ^= block[r, 0]

            block ^= rk_L2[idx:idx+16].reshape(4, 4)
            idx += 16

            block = INV_SBOX_L2[block]
            L2_dec[i:i+4, j:j+4] = block

    return L2_dec


def decrypt_layer3(enc_img):
    return enc_img ^ stream_L3

# =============================================================================
# RESILIENCE TEST
# =============================================================================
def test_resilience(attacked_img):
    try:
        img_enc_layers_rec = global_diffusion_decrypt(attacked_img, K_GLOBAL)

        L1_enc_rec = img_enc_layers_rec ^ L2_enc ^ L3_enc
        L2_enc_rec = img_enc_layers_rec ^ L1_enc ^ L3_enc
        L3_enc_rec = img_enc_layers_rec ^ L1_enc ^ L2_enc

        L1_rec = decrypt_layer1(L1_enc_rec)
        L2_rec = decrypt_layer2(L2_enc_rec)
        L3_rec = decrypt_layer3(L3_enc_rec)

        img_rec_full = (L1_rec | L2_rec | L3_rec).astype(np.uint8)

        img_rec = cv2.resize(
            img_rec_full,
            (img.shape[1], img.shape[0]),
            interpolation=cv2.INTER_NEAREST
        )

        return img_rec, img_rec_full

    except:
        return None, None

# =============================================================================
# ATTACK CONFIG
# =============================================================================
attack_configs = {
    "Salt&Pepper 5%": lambda x: add_salt_pepper_noise(x, 0.05),
    "Salt&Pepper 10%": lambda x: add_salt_pepper_noise(x, 0.10),
    "Gaussian σ=20": lambda x: add_gaussian_noise(x, 0, 20),
    "JPEG Q=50": lambda x: jpeg_compression_attack(x, 50),
    "Crop 10%": lambda x: crop_attack(x, 0.10),
}

# =============================================================================
# RUN TESTS (NO PRINTS, ONLY TABLE)
# =============================================================================
results = []

for name, attack_fn in attack_configs.items():
    img_attacked = attack_fn(img_cipher)
    img_rec_resized, _ = test_resilience(img_attacked)

    if img_rec_resized is not None:
        mse = np.mean((img - img_rec_resized) ** 2)
        psnr = 10 * np.log10(255**2 / mse) if mse > 0 else 100
        ssim_val = ssim(img, img_rec_resized, data_range=255)

        results.append([
            name,
            round(psnr, 2),
            round(ssim_val, 4),
            round(mse, 2)
        ])

# =============================================================================
# FINAL OUTPUT
# =============================================================================
df = pd.DataFrame(results, columns=['Attack', 'PSNR(dB)', 'SSIM', 'MSE'])
print(df.to_string(index=False))